<a href="https://colab.research.google.com/github/safaabuzaid/mri-generalization/blob/main/02_Baseline_Comparison_3Seeds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Model Comparison Under Cross-Dataset Domain Shift

## Objective

Evaluate the cross-domain generalization of three pretrained CNN architectures:

- ResNet18
- DenseNet121
- EfficientNet-B3

Models are trained and validated on Dataset A (figshare dataset) and evaluated on an independent Dataset B (SARTAJ dataset).

Each experiment is repeated using three random seeds to assess the stability of the results.

### Research question

**Which architecture provides the most reliable generalization from Dataset A to an independent Dataset B?**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
src_path = '/content/drive/MyDrive/MRI_Generalization/src'

sys.path.append(src_path)

In [ ]:
import os
import random
import torch
import pandas as pd
from pandas import DataFrame
import numpy as np
from data import BrainTumorDataset
from data_preparation import split_dataset, create_dataloaders, create_dataset
from train import train_model
from evaluate import test, create_confusion_matrix, classification_report
from transform import baseline_transform, augmentation_transform
from models import (
    get_effecientnet_b3,
    get_resnet18,
    get_densenet121
)
from torch.utils.data import random_split

In [ ]:
#Reproducibility

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [42, 123, 2026]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
# Configuration

BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_CLASSES = 3

IMAGE_SIZE = 300

LEARNING_RATE = 0.001

### Prepare datasets

In [ ]:
SPLIT_DIR = "/content/drive/MyDrive/MRI_Generalization/splits"

train_df = pd.read_csv("/content/drive/MyDrive/MRI_Generalization/project_data/splits/train_split.csv")
val_df = pd.read_csv("/content/drive/MyDrive/MRI_Generalization/project_data/splits/val_split.csv")
test_df = pd.read_csv("/content/drive/MyDrive/MRI_Generalization/project_data/splits/test_split.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("External Test:", len(test_df))

Train: 2451
Validation: 613
External Test: 2543


In [ ]:
print(train_df.head())
print(val_df.head())
print(test_df.head())

                                                path class_name  label
0  /content/drive/MyDrive/MRI_Generalization/proj...     glioma      0
1  /content/drive/MyDrive/MRI_Generalization/proj...     glioma      0
2  /content/drive/MyDrive/MRI_Generalization/proj...     glioma      0
3  /content/drive/MyDrive/MRI_Generalization/proj...     glioma      0
4  /content/drive/MyDrive/MRI_Generalization/proj...     glioma      0
                                                path  class_name  label
0  /content/drive/MyDrive/MRI_Generalization/proj...      glioma      0
1  /content/drive/MyDrive/MRI_Generalization/proj...      glioma      0
2  /content/drive/MyDrive/MRI_Generalization/proj...  meningioma      1
3  /content/drive/MyDrive/MRI_Generalization/proj...  meningioma      1
4  /content/drive/MyDrive/MRI_Generalization/proj...  meningioma      1
                                                path class_name  label
0  /content/drive/MyDrive/MRI_Generalization/proj...     glioma      0


In [ ]:
# create datasets

train_dataset = BrainTumorDataset(
    dataframe=train_df,
    transform=baseline_transform
)

val_dataset = BrainTumorDataset(
    dataframe=val_df,
    transform=baseline_transform
)

test_dataset = BrainTumorDataset(
    dataframe=test_df,
    transform=baseline_transform
)

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, val_dataset, test_dataset)

### Define the Experiment

In [ ]:
models_to_test = {
    "ResNet18": get_resnet18,
    "DenseNet121": get_densenet121,
    "EfficientNet-B3": get_effecientnet_b3
}

In [ ]:
results = []

### Run the experiments

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
for model_name, model_function in models_to_test.items():

    for seed in SEEDS:

        print("=" * 60)
        print(f"Model: {model_name}")
        print(f"Seed: {seed}")
        print("=" * 60)

        set_seed(seed)

        model, criterion, optimizer, device = model_function()

        # Train
        (
        history,
        best_epoch,
        best_train_loss,
        best_train_acc,
        best_val_loss,
        best_val_acc
        ) = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            num_epochs=NUM_EPOCHS
        )

        # Evaluate external dataset
        all_labels, all_preds = test(
            model,
            test_loader,
            device
        )

        test_accuracy = accuracy_score(all_labels, all_preds)

        macro_precision = precision_score(
            all_labels,
            all_preds,
            average="macro"
        )

        macro_recall = recall_score(
            all_labels,
            all_preds,
            average="macro"
        )

        macro_f1 = f1_score(
            all_labels,
            all_preds,
            average="macro"
        )

        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Macro Precision: {macro_precision:.4f}")
        print(f"Macro Recall: {macro_recall:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")

        results.append({
            "model": model_name,
            "seed": seed,

            "best_epoch": best_epoch,

            "train_loss": best_train_loss,
            "train_acc": best_train_acc,

            "val_loss": best_val_loss,
            "val_acc": best_val_acc,

            "test_accuracy": test_accuracy,
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1
        })

        # confusion matrices
        cm = confusion_matrix(all_labels, all_preds)

        cm_df = pd.DataFrame(
            cm,
            index=["Glioma", "Meningioma", "Pituitary"],
            columns=["Glioma", "Meningioma", "Pituitary"]
        )

        cm_df.to_csv(
            f"cm_{model_name}_{seed}.csv"
        )

        #classification reports

        report = classification_report(
            all_labels,
            all_preds,
            target_names=[
                "Glioma",
                "Meningioma",
                "Pituitary"
            ],
            output_dict=True
        )

        report_df = pd.DataFrame(report).transpose()

        report_df.to_csv(
            f"classification_report_{model_name}_{seed}.csv"
        )





Model: ResNet18
Seed: 42
Best model saved!
Epoch [1/10]
Train Loss: 0.3349, Train Acc: 0.8617
Val Loss: 0.2582, Val Acc: 0.9152
------------------------------
Epoch [2/10]
Train Loss: 0.1589, Train Acc: 0.9425
Val Loss: 0.3839, Val Acc: 0.8483
------------------------------
Best model saved!
Epoch [3/10]
Train Loss: 0.1269, Train Acc: 0.9527
Val Loss: 0.1447, Val Acc: 0.9396
------------------------------
Best model saved!
Epoch [4/10]
Train Loss: 0.0779, Train Acc: 0.9731
Val Loss: 0.1714, Val Acc: 0.9445
------------------------------
Epoch [5/10]
Train Loss: 0.0620, Train Acc: 0.9780
Val Loss: 0.3215, Val Acc: 0.8728
------------------------------
Epoch [6/10]
Train Loss: 0.0697, Train Acc: 0.9743
Val Loss: 0.4418, Val Acc: 0.8956
------------------------------
Epoch [7/10]
Train Loss: 0.0519, Train Acc: 0.9800
Val Loss: 0.3759, Val Acc: 0.8728
------------------------------
Best model saved!
Epoch [8/10]
Train Loss: 0.0487, Train Acc: 0.9853
Val Loss: 0.1461, Val Acc: 0.9462
------

100%|██████████| 30.8M/30.8M [00:00<00:00, 151MB/s]


Best model saved!
Epoch [1/10]
Train Loss: 0.2841, Train Acc: 0.8858
Val Loss: 4.2476, Val Acc: 0.4046
------------------------------
Best model saved!
Epoch [2/10]
Train Loss: 0.1627, Train Acc: 0.9453
Val Loss: 1.6809, Val Acc: 0.5938
------------------------------
Best model saved!
Epoch [3/10]
Train Loss: 0.1464, Train Acc: 0.9449
Val Loss: 0.3751, Val Acc: 0.8728
------------------------------
Best model saved!
Epoch [4/10]
Train Loss: 0.1352, Train Acc: 0.9498
Val Loss: 0.1528, Val Acc: 0.9462
------------------------------
Epoch [5/10]
Train Loss: 0.1101, Train Acc: 0.9551
Val Loss: 0.2023, Val Acc: 0.9282
------------------------------
Best model saved!
Epoch [6/10]
Train Loss: 0.0469, Train Acc: 0.9853
Val Loss: 0.1352, Val Acc: 0.9478
------------------------------
Epoch [7/10]
Train Loss: 0.0566, Train Acc: 0.9800
Val Loss: 0.2401, Val Acc: 0.9282
------------------------------
Epoch [8/10]
Train Loss: 0.0779, Train Acc: 0.9739
Val Loss: 0.6778, Val Acc: 0.8042
-------------

100%|██████████| 47.2M/47.2M [00:00<00:00, 172MB/s]


Best model saved!
Epoch [1/10]
Train Loss: 0.5420, Train Acc: 0.7911
Val Loss: 0.2502, Val Acc: 0.9184
------------------------------
Best model saved!
Epoch [2/10]
Train Loss: 0.1842, Train Acc: 0.9429
Val Loss: 0.1449, Val Acc: 0.9608
------------------------------
Best model saved!
Epoch [3/10]
Train Loss: 0.0770, Train Acc: 0.9759
Val Loss: 0.0749, Val Acc: 0.9690
------------------------------
Best model saved!
Epoch [4/10]
Train Loss: 0.0445, Train Acc: 0.9878
Val Loss: 0.0608, Val Acc: 0.9739
------------------------------
Best model saved!
Epoch [5/10]
Train Loss: 0.0256, Train Acc: 0.9943
Val Loss: 0.0528, Val Acc: 0.9772
------------------------------
Epoch [6/10]
Train Loss: 0.0198, Train Acc: 0.9955
Val Loss: 0.0490, Val Acc: 0.9755
------------------------------
Best model saved!
Epoch [7/10]
Train Loss: 0.0153, Train Acc: 0.9959
Val Loss: 0.0624, Val Acc: 0.9788
------------------------------
Epoch [8/10]
Train Loss: 0.0098, Train Acc: 0.9967
Val Loss: 0.1029, Val Acc: 0.

In [ ]:
print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=[
            "Glioma",
            "Meningioma",
            "Pituitary"
        ],
        digits=4
    )
)

In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "baseline_3seeds_raw_results.csv",
    index=False
)

print("Results saved!")

In [ ]:
print(type(results))
print(len(results))


NameError: name 'results' is not defined